# 🤖 Module 1 — Lời giải (solution)

> 📘 **File này chứa lời giải tham chiếu cho 3 bài tập của Module 1.**
> Hãy tự làm trong `module_1_ex.ipynb` trước, sau đó đối chiếu ở đây.

---


In [2]:
# --- Setup: chạy cell này một lần duy nhất ---
# Cài đặt thư viện (bỏ comment nếu bạn chạy lần đầu):
# !pip install -U transformers accelerate matplotlib
import os
import time
import random
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import logging
import sys

sys.stderr = open(os.devnull, 'w')
logging.set_verbosity_error()
torch.set_num_threads(6)

[transformers] The model 'OptimizedModule' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMamba

In [3]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from transformers.pipelines import pipeline
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
print(f"Loading {MODEL_NAME} on CPU ... (lần đầu có thể mất 1–2 phút)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cpu",
    torch_dtype="auto",
    trust_remote_code=False,
    attn_implementation="sdpa" 
)

model = torch.compile(model)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)
print("✅ Model sẵn sàng!")

Loading Qwen/Qwen2-0.5B-Instruct on CPU ... (lần đầu có thể mất 1–2 phút)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Model sẵn sàng!


---

## ✅ Phần 2 — Lời giải

Bên dưới là lời giải hoàn chỉnh cho từng bài tập.
Mỗi bài gồm: đề bài (giữ nguyên) + code giải đã điền đầy đủ.


### Bài tập 1 — Hallucination Detector

#### 🎯 Mục tiêu
- Thiết kế một "detector" đơn giản: hỏi LLM, rồi kiểm tra câu trả lời có chứa keyword đúng không.
- Trực tiếp trải nghiệm LLM **tự tin trả lời sai** với câu hỏi không có đáp án thực.

#### 📝 Hướng dẫn thực hiện
**TODO** — Duyệt qua tất cả expected_keywords nếu có bất kỳ keyword nào trong answer thì trả ra False còn không ra True.


In [ ]:
def ask_llm(question: str) -> str:
    # 1a: gọi generator greedy, trả về text đã strip
    out = generator(question, max_new_tokens=30,do_sample=False)[0]["generated_text"]
    return out.strip()

def is_hallucination(answer: str, expected_keywords: list) -> bool:
    """True nếu answer không chứa keyword nào, hoặc list rỗng (câu hỏi bịa)."""

    # 1b: nếu không tìm thấy bất kỳ keyword nào → hallucination
    return not any(kw.lower() in answer.lower() for kw in expected_keywords)


In [4]:
# Kiểm tra kết quả

# 3 câu: 1 fact thật + 2 bịa
questions = [
    ("Thủ đô của Việt Nam là gì? Trả lời ngắn gọn trong 1 câu",              ["hà nội", "hanoi"]),
    ("Việt nam có bao nhiêu đầu đạn hạt nhân? Trả lời ngắn gọn trong 1 câu", ['không biết','không','chưa']),
    ("Ai phát minh máy tính lượng tử năm 1850? Trả lời ngắn gọn trong 1 câu",['không biết','không','chưa']),
]  # Giá trị của list là các tuple gồm 2 phần : (question, expected_keywords)


# Duyệt qua  (question, expected_keywords) trong list -> gọi llm trả lời question
#            -> kiểm tra hallucination
for q, kws in questions:
    ans = ask_llm(q)
    is_h = is_hallucination(ans, kws)
    tag = "🚨 Hallucination" if is_h else "✅ OK"
    print(f"[{tag}]  Q: {q}")
    print(f"         A: {ans}\n")

[✅ OK]  Q: Thủ đô của Việt Nam là gì? Trả lời ngắn gọn trong 1 câu
         A: : Thủ đô của Việt Nam là Hà Nội. 

Để trả lời đúng, tôi cần biết thêm thông tin về thủ đô của Việt Nam.

[🚨 Hallucination]  Q: Việt nam có bao nhiêu đầu đạn hạt nhân? Trả lời ngắn gọn trong 1 câu
         A: : Việt Nam có 20 đầu đạn hạt nhân. 

Để trả lời câu này, tôi cần biết thông tin về số lượng đầu

[🚨 Hallucination]  Q: Ai phát minh máy tính lượng tử năm 1850? Trả lời ngắn gọn trong 1 câu
         A: "Giữa năm 1850, máy tính lượng tử được phát minh bởi Arthur Eddington."



### Bài tập 3 — Sentiment Analyzer bằng Prompt

#### 🎯 Mục tiêu
- Dùng LLM như một **classifier không cần training**: chỉ cần viết prompt đúng.
- Biết cách thiết kế prompt để LLM trả về output có cấu trúc (1 nhãn duy nhất).


#### 📝 Hướng dẫn thực hiện
1. **TODO 2a** — Xây dựng `prompt` hướng dẫn LLM phân loại thành `'tích cực'`, `'tiêu cực'`, hoặc `'trung lập'`, **chỉ trả 1 cụm, không giải thích**.
2. **TODO 2b** — Kiểm tra label có nằm trong kết quả trả về của model hay không.


In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from transformers.pipelines import pipeline
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
print(f"Loading {MODEL_NAME} on CPU ... (lần đầu có thể mất 1–2 phút)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cpu",
    torch_dtype="auto",
    trust_remote_code=False,
    attn_implementation="sdpa" 
)

model = torch.compile(model)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)
print("✅ Model sẵn sàng!")

In [18]:
import re

def clear_reasoning(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def analyze_sentiment(text: str) -> str:
    """Dùng LLM phân loại cảm xúc: 'tích cực', 'tiêu cực', hoặc 'trung lập'."""

    # 2a: prompt rõ ràng, yêu cầu chỉ trả 1 cụm, không giải thích
    prompt = (
        f"Phân loại cảm xúc câu sau thành 'tích cực', 'tiêu cực', hoặc 'trung lập'. Trả lời ngắn gọn trong 1 cụm ('tích cực', 'tiêu cực', hoặc 'trung lập'), không giải thích gì thêm.\n\n"
        f"Câu: {text} \n"
    )
    raw =clear_reasoning(generator(prompt, max_new_tokens=32,top_p=0.95,temperature=0.3,do_sample=True)[0]["generated_text"].lower().strip())

    for label in ["tích cực", "tiêu cực", "trung lập"]:
        if label in raw:   # 2b: kiểm tra label có trong response không
            return label
    return "không rõ"

# Chạy thử 4 câu
samples = [
    "Sản phẩm này quá tệ.",
    "Hôm nay tôi rất vui vì vừa pass được bài tập khó!",
    "Hôm nay trời có mây, nhiệt độ 28 độ.",
    "Ứng dụng này siêu tiện lợi, tôi yêu nó!",
]
labels = []
for s in samples:
    label = analyze_sentiment(s)
    labels.append(label)
    print(f"{label:>12} | {s}")


    tiêu cực | Sản phẩm này quá tệ.
    tích cực | Hôm nay tôi rất vui vì vừa pass được bài tập khó!
   trung lập | Hôm nay trời có mây, nhiệt độ 28 độ.
    tích cực | Ứng dụng này siêu tiện lợi, tôi yêu nó!


---

## 🏁 Kết thúc Module 1

Chúc mừng! Đến đây bạn đã:

- ✅ Phân biệt **deterministic vs probabilistic** và hiểu đây là thay đổi tư duy lớn nhất khi chuyển sang lập trình với LLM.
- ✅ Hiểu cơ chế **dự đoán token tiếp theo**: text → logits → softmax → 1 token — và vòng lặp autoregressive.
- ✅ **Tự xây vòng lặp autoregressive** bằng Python thuần, không cần biết PyTorch.
- ✅ Trải nghiệm **hallucination** và có baseline detector đơn giản bằng keyword matching.
- ✅ Dùng LLM như **classifier không cần training** — chỉ qua prompt engineering.
